# Boundary Viewer
This notebook loads the rectangular boundary definitions and plots them on a world map.

The map is split by class:
- Countries
- Cities
- Regions

Use this to inspect the geometry before you pass a box into the FIRMS filtering code.

In [13]:
from pathlib import Path
import json
import folium
from IPython.display import display

BOUNDARY_FILE = Path('country_rectangles.json')

with BOUNDARY_FILE.open('r', encoding='utf-8') as handle:
    boundary_data = json.load(handle)

boundary_data['crs']

'EPSG:4326'

In [14]:
def make_shape_feature(entry, color, fill_opacity=0.12):
    if 'polygon' in entry:
        bounds = [[lat, lon] for lon, lat in entry['polygon']]
        return folium.Polygon(
            locations=bounds,
            color=color,
            weight=2,
            fill=True,
            fill_color=color,
            fill_opacity=fill_opacity,
            tooltip=f"{entry['name']} | {entry['type']}"
        )

    west, south, east, north = entry['bbox']
    return folium.Rectangle(
        bounds=[[south, west], [north, east]],
        color=color,
        weight=2,
        fill=True,
        fill_color=color,
        fill_opacity=fill_opacity,
        tooltip=f"{entry['name']} | {entry['type']}"
    )

def build_map(entries, title):
    world = folium.Map(location=[20, 20], zoom_start=2, tiles='CartoDB positron')
    colors = {
        'country': '#1f77b4',
        'city': '#d62728',
        'region': '#2ca02c',
    }

    grouped = {}
    for entry in entries:
        grouped.setdefault(entry['type'], []).append(entry)

    for group_name in ['country', 'city', 'region']:
        feature_group = folium.FeatureGroup(name=group_name.title(), show=True)
        for entry in grouped.get(group_name, []):
            shape = make_shape_feature(entry, colors[group_name])
            shape.add_to(feature_group)
        feature_group.add_to(world)

    folium.LayerControl(collapsed=False).add_to(world)
    return world

all_entries = boundary_data['entries']
world_map = build_map(all_entries, 'All Boundaries')
world_map

In [15]:
def build_class_maps(entries):
    maps = {}
    for group_name, zoom in [('country', 3), ('city', 5), ('region', 3)]:
        group_entries = [entry for entry in entries if entry['type'] == group_name]
        if not group_entries:
            continue
        center_lat = sum((entry['bbox'][1] + entry['bbox'][3]) / 2 for entry in group_entries) / len(group_entries)
        center_lon = sum((entry['bbox'][0] + entry['bbox'][2]) / 2 for entry in group_entries) / len(group_entries)
        m = folium.Map(location=[center_lat, center_lon], zoom_start=zoom, tiles='CartoDB positron')
        color = {'country': '#1f77b4', 'city': '#d62728', 'region': '#2ca02c'}[group_name]
        for entry in group_entries:
            make_shape_feature(entry, color).add_to(m)
        maps[group_name] = m
    return maps

class_maps = build_class_maps(all_entries)
for name, map_object in class_maps.items():
    print(f'--- {name.upper()} ---')
    display(map_object)

--- COUNTRY ---


--- CITY ---


--- REGION ---


In [16]:
# Optional: save the maps to HTML files in case you want to open them outside the notebook.
output_dir = Path('html_maps')
output_dir.mkdir(exist_ok=True)
world_map.save(output_dir / 'all_boundaries_map.html')
for name, map_object in class_maps.items():
    map_object.save(output_dir / f'{name}_boundaries_map.html')
print(f'Saved HTML maps to {output_dir.resolve()}')

Saved HTML maps to C:\.Okul\CE49X Introduction to Computational Thinking and Data Science for Civil Engineers\GithubRepository\CE49X\Final Project\boundaries\html_maps
